# GLM-OCR: greedy versus beam-four validation

Compares **greedy (`num_beams=1`) and beam search (`num_beams=4`)** on the best saved curriculum checkpoint and the same full development holdout. It preserves the native OCR prompt, spatial resize, processor defaults, generation limit and explicit stop tokens from the saved training configuration.

This notebook performs **inference only**. It does not train, merge weights, change the training notebooks, create a submission or replace the existing final selection. Run setup first, then the clearly marked full-validation cell. Predictions are cached after each image/mode, so interruptions are resumable. Reports are written into a separate comparison directory.

Both modes are generated afresh for this experiment; previously saved greedy predictions are used only as a reproducibility reference, not as a timing baseline. The full comparison requires a CUDA GPU and takes longer than greedy validation alone. Creation-time checks use only a small bounded sample, not the full validation loop.

In [ ]:
import gc, json, math, hashlib, time, unicodedata, platform, sys
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import torch, transformers, peft
from transformers import AutoProcessor, GlmOcrForConditionalGeneration
from peft import PeftModel
from PIL import Image, ImageOps
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from IPython.display import display

ROOT=Path.cwd().resolve()
assert (ROOT/'resources/Train.csv').is_file(), 'Open with the project root as working directory.'
RUN=ROOT/'training_outputs/glm_ocr_from_base_curriculum_v7_native_ocr_balanced'
BASE_MODEL=ROOT/'model/glm-ocr'; IMAGES=ROOT/'resources/images'
BEAM_SIZES=(1,4)
BEAM_LENGTH_PENALTY=1.0
BEAM_EARLY_STOPPING=True
BOOTSTRAP_REPLICATES=2000
saved=json.loads((RUN/'run_identity.json').read_text(encoding='utf-8'))
CONFIG=saved['identity']['config']
assert CONFIG['prompt']=='Text Recognition:'
WORD_NORMALIZER=CONFIG['word_normalizer']; CHARACTER_NORMALIZER=CONFIG['character_normalizer']
DEVICE='cuda:0'
DTYPE=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
normalize=lambda s:unicodedata.normalize('NFC',str(s)).strip()
pd.set_option('display.max_colwidth',180)

def file_hash(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(8*1024*1024),b''):digest.update(block)
    return digest.hexdigest()
def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def write_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8');temp.replace(path)

candidates=[]
for stage in ['stage1','stage2']:
    path=RUN/stage/'history.json'
    if path.exists():
        for row in json.loads(path.read_text(encoding='utf-8')):
            stem='incoming' if row['step']==0 else f"step-{row['step']}"
            candidates.append({'stage':stage,**row,'predictions_path':RUN/stage/'validation'/f'{stem}_predictions.csv'})
assert candidates and all(math.isfinite(float(x['metrics']['zindi_score_proxy'])) for x in candidates)
selected=max(candidates,key=lambda x:x['metrics']['zindi_score_proxy'])
SPEC=selected['spec']
MODEL_PATH=BASE_MODEL if SPEC['kind']=='base' else RUN/SPEC['path']
assert MODEL_PATH.exists()
validation=pd.read_csv(RUN/'validation_manifest.csv',dtype={'ID':str},keep_default_na=False).sort_values('ID').reset_index(drop=True)
audit=pd.read_csv(RUN/'split_and_label_audit.csv',dtype={'ID':str},keep_default_na=False)
assert validation.ID.is_unique and len(validation)==CONFIG['holdout_size']
assert set(validation.ID)==set(audit.loc[audit.split.eq('validation'),'ID'])
assert set(validation.ID).isdisjoint(audit.loc[audit.split.eq('train'),'ID'])
assert set(validation.ID).isdisjoint(saved['identity']['excluded_ids'])
assert validation.set_index('ID').Target.sort_index().equals(audit.loc[audit.split.eq('validation')].set_index('ID').Target.sort_index())
print('Selected:',SPEC,'saved greedy proxy:',selected['metrics']['zindi_score_proxy'])
print('Validation images:',len(validation),'| Decoders:',BEAM_SIZES)

## Shared preprocessing and decoding

One encoder serves both modes. Images are EXIF-corrected, converted to RGB and resized to the same spatial pixel bounds, followed by the base processor's patch rounding. The prompt ends at the plain assistant header. Sampling and repetition penalties are not introduced. Beam-four uses length penalty 1.0 and early stopping enabled.

Timing uses CUDA synchronization around `generate`, excluding preprocessing, disk I/O and warmup. Mode order alternates by image. Measurements are approximate and can vary with GPU load/temperature; total times are sums of recorded generation calls, including calls from previous resumed sessions.

In [ ]:
processor=AutoProcessor.from_pretrained(BASE_MODEL,local_files_only=True,use_fast=False)
base_generation=json.loads((BASE_MODEL/'generation_config.json').read_text(encoding='utf-8'))
base_eos=base_generation['eos_token_id']
base_eos=[base_eos] if isinstance(base_eos,int) else list(base_eos)
EOS_IDS=sorted(set(base_eos+[processor.tokenizer.eos_token_id]))

def resize_spatial(image):
    image=image.convert('RGB')
    for bound,comparison in [(CONFIG['max_spatial_pixels'],lambda area,x:area>x),
                             (CONFIG['min_spatial_pixels'],lambda area,x:area<x)]:
        if comparison(image.width*image.height,bound):
            scale=math.sqrt(bound/(image.width*image.height))
            image=image.resize((max(1,int(image.width*scale)),max(1,int(image.height*scale))),Image.Resampling.BICUBIC)
    return image

def encode_image(image_id):
    with Image.open(IMAGES/f'{image_id}.jpg') as source:
        image=resize_spatial(ImageOps.exif_transpose(source).convert('RGB'))
    text=('[gMASK]<sop><|user|>\n<|begin_of_image|><|image|><|end_of_image|>'+CONFIG['prompt']+'<|assistant|>')
    inputs=processor(text=[text],images=[image],add_special_tokens=False,return_tensors='pt')
    inputs.pop('token_type_ids',None)
    assert int(inputs['input_ids'][0,-1])==processor.tokenizer.convert_tokens_to_ids('<|assistant|>')
    assert 'mm_token_type_ids' in inputs
    assert torch.equal(inputs['mm_token_type_ids'].eq(1),inputs['input_ids'].eq(processor.tokenizer.convert_tokens_to_ids('<|image|>')))
    return inputs

def generation_options(beams):
    assert beams in BEAM_SIZES
    options={'max_new_tokens':CONFIG['max_new_tokens'],'num_beams':beams,'num_return_sequences':1,
             'do_sample':False,'use_cache':True,'eos_token_id':EOS_IDS,
             'pad_token_id':processor.tokenizer.pad_token_id}
    if beams>1:options.update(length_penalty=BEAM_LENGTH_PENALTY,early_stopping=BEAM_EARLY_STOPPING)
    return options

def repeat_flag(text):
    # Diagnostic only: a three-word sequence occurring at least three times.
    words=text.casefold().split()
    counts=Counter(tuple(words[i:i+3]) for i in range(max(0,len(words)-2)))
    return max(counts.values(),default=0)>=3

def load_model():
    assert torch.cuda.is_available(), 'CUDA GPU required for this inference comparison.'
    gc.collect();torch.cuda.empty_cache()
    model=GlmOcrForConditionalGeneration.from_pretrained(BASE_MODEL,local_files_only=True,
        dtype=DTYPE,device_map={'':0},attn_implementation='sdpa')
    if SPEC['kind']=='adapter':model=PeftModel.from_pretrained(model,MODEL_PATH,is_trainable=False)
    model.eval()
    for parameter in model.parameters():parameter.requires_grad_(False)
    return model

def decode_once(model,inputs,beams):
    torch.cuda.synchronize()
    start=time.perf_counter()
    with torch.inference_mode():output=model.generate(**inputs,**generation_options(beams))
    torch.cuda.synchronize()
    seconds=time.perf_counter()-start
    ids=output[0,inputs['input_ids'].shape[1]:].tolist()
    # Beam output can include padding after the first EOS; exclude it from lengths.
    end=next((i for i,token in enumerate(ids) if token in EOS_IDS),None)
    if end is not None:ids=ids[:end+1]
    prediction=normalize(processor.decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False))
    return {'Prediction':prediction,'generated_tokens':len(ids),'stopped_with_eos':end is not None,
        'hit_token_limit':end is None and len(ids)>=CONFIG['max_new_tokens'],
        'repetition_flag':repeat_flag(prediction),'generation_seconds':seconds}

## Experiment identity and safe resume

Cache identity includes base and adapter file contents, source code, input image hashes, validation labels, decoding options, package versions and GPU identity. An interrupted run reuses only matching completed image/mode records. There is no automatic reduction of resolution or beam size after an out-of-memory error, since that would invalidate the comparison.

In [ ]:
notebook_path=ROOT/'notebooks/glm_ocr_greedy_vs_beam4_validation.ipynb'
notebook=json.loads(notebook_path.read_text(encoding='utf-8'))
code_hash=object_hash([''.join(c['source']) for c in notebook['cells'] if c['cell_type']=='code'])
versions={'torch':torch.__version__,'transformers':transformers.__version__,'peft':peft.__version__,
          'numpy':np.__version__,'pandas':pd.__version__,'python':platform.python_version()}
image_hashes={image_id:file_hash(IMAGES/f'{image_id}.jpg') for image_id in tqdm(validation.ID,desc='Fingerprint validation images')}
model_files=sorted(set([p for folder in {BASE_MODEL,MODEL_PATH} for p in folder.iterdir()
                       if p.suffix in {'.json','.safetensors','.jinja','.model'}]))
identity={'model':SPEC,'model_files':{str(p.relative_to(ROOT)):file_hash(p) for p in model_files},
    'validation':file_hash(RUN/'validation_manifest.csv'),'images':image_hashes,
    'source':code_hash,'versions':versions,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'dtype':str(DTYPE),'prompt':CONFIG['prompt'],'min_pixels':CONFIG['min_spatial_pixels'],
    'max_pixels':CONFIG['max_spatial_pixels'],'generation':{str(b):generation_options(b) for b in BEAM_SIZES}}
EXPERIMENT_KEY=object_hash(identity)
OUT=RUN/'greedy_vs_beam4'/EXPERIMENT_KEY[:16]
OUT.mkdir(parents=True,exist_ok=True)
manifest=OUT/'experiment.json'
if manifest.exists():assert json.loads(manifest.read_text(encoding='utf-8'))==identity
else:write_json(manifest,identity)
print('Experiment output:',OUT)

def cache_path(image_id,beams):return OUT/'cache'/f'beam{beams}'/f'{image_id}.json'
def read_cached(image_id,beams):
    path=cache_path(image_id,beams)
    if not path.exists():return None
    row=json.loads(path.read_text(encoding='utf-8'))
    assert row['experiment_key']==EXPERIMENT_KEY and row['image_hash']==image_hashes[image_id]
    assert row['ID']==image_id and row['num_beams']==beams
    assert isinstance(row['Prediction'],str) and row['generation_seconds']>=0
    return row

def run_comparison(frame):
    assert frame.ID.is_unique and set(frame.ID).issubset(set(validation.ID))
    missing=[(image_id,b) for image_id in frame.ID for b in BEAM_SIZES if read_cached(image_id,b) is None]
    if not missing:
        print('All requested predictions already cached.');return
    model=load_model()
    try:
        warm=encode_image(frame.iloc[0].ID).to(DEVICE)
        for beams in BEAM_SIZES:decode_once(model,warm,beams)
        del warm
        for index,row in enumerate(tqdm(frame.itertuples(),total=len(frame),desc='Validation: greedy + beam4')):
            order=BEAM_SIZES if index%2==0 else BEAM_SIZES[::-1]
            pending=[b for b in order if read_cached(row.ID,b) is None]
            if not pending:continue
            inputs=encode_image(row.ID).to(DEVICE)
            for beams in pending:
                try:
                    result=decode_once(model,inputs,beams)
                except torch.cuda.OutOfMemoryError:
                    print(f'GPU memory exhausted at {row.ID}, beam={beams}. Completed predictions remain cached. '
                          'Free GPU memory and rerun; no settings were silently changed.')
                    raise
                write_json(cache_path(row.ID,beams),{'experiment_key':EXPERIMENT_KEY,'image_hash':image_hashes[row.ID],
                    'ID':row.ID,'num_beams':beams,**result})
            del inputs
    finally:
        del model;gc.collect();torch.cuda.empty_cache()

## RUN FULL VALIDATION — inference only

This cell runs both decoders for all validation images (currently 400 × 2 calls, plus warmup). Stop and rerun it to resume from the cache. Run it before the reporting cells; partial results cannot be used to select a decoder.

In [ ]:
run_comparison(validation)

## Accuracy, failure counts and inference time

The local proxy uses the same length-weighted edit counts as training; standard WER/CER are also reported. Exact match includes a raw comparison and an NFC/edge-trimmed comparison. Repetition and >1.5× output-length flags are heuristics for review, not proof of hallucination. The repetition count below excludes references that themselves trigger the heuristic.

In [ ]:
def add_metrics(frame):
    frame=frame.copy()
    frame['raw_exact']=frame.Target.eq(frame.Prediction)
    frame['Target']=frame.Target.map(normalize);frame['Prediction']=frame.Prediction.map(normalize)
    frame['nwords']=frame.Target.str.split().str.len().clip(lower=1)
    frame['nchars']=frame.Target.str.len().clip(lower=1)
    frame['word_edits']=[Levenshtein.distance(a.split(),b.split()) for a,b in zip(frame.Target,frame.Prediction)]
    frame['char_edits']=[Levenshtein.distance(a,b) for a,b in zip(frame.Target,frame.Prediction)]
    frame['exact']=frame.Target.eq(frame.Prediction)
    frame['new_repetition_flag']=frame.repetition_flag & ~frame.Target.map(repeat_flag)
    frame['length_flag']=(frame.Prediction.str.len()/frame.nchars).gt(1.5)
    return frame

def score(frame):
    ew=np.dot(frame.word_edits,np.sqrt(frame.nwords))/np.sqrt(frame.nwords).sum()
    ec=np.dot(frame.char_edits,np.sqrt(frame.nchars))/np.sqrt(frame.nchars).sum()
    return float(1-.5*(ew/WORD_NORMALIZER+ec/CHARACTER_NORMALIZER))

def summarize(frame):
    return {'images':len(frame),'zindi_proxy':score(frame),
        'WER_percent':100*frame.word_edits.sum()/frame.nwords.sum(),
        'CER_percent':100*frame.char_edits.sum()/frame.nchars.sum(),
        'raw_exact_lines':int(frame.raw_exact.sum()),'normalized_exact_lines':int(frame.exact.sum()),
        'blank_outputs':int(frame.Prediction.eq('').sum()),'token_limit_hits':int(frame.hit_token_limit.sum()),
        'new_repetition_flags':int(frame.new_repetition_flag.sum()),'length_flags':int(frame.length_flag.sum()),
        'mean_generation_seconds':float(frame.generation_seconds.mean()),
        'median_generation_seconds':float(frame.generation_seconds.median()),
        'total_generation_seconds':float(frame.generation_seconds.sum())}

results={}
for beams in BEAM_SIZES:
    records=[read_cached(image_id,beams) for image_id in validation.ID]
    assert all(x is not None for x in records), 'Run full validation before computing comparison results.'
    result=validation.merge(pd.DataFrame(records),on='ID',validate='one_to_one')
    assert len(result)==CONFIG['holdout_size']
    result=add_metrics(result)
    results[beams]=result
    result.to_csv(OUT/f'beam{beams}_predictions.csv',index=False)
summary=pd.DataFrame([{'num_beams':b,**summarize(results[b])} for b in BEAM_SIZES]).set_index('num_beams')
summary['relative_generation_time']=summary.total_generation_seconds/summary.loc[1,'total_generation_seconds']
display(summary.T);summary.to_csv(OUT/'summary.csv')

old=pd.read_csv(selected['predictions_path'],dtype={'ID':str},keep_default_na=False).set_index('ID')
assert set(old.index)==set(validation.ID)
fresh=results[1].set_index('ID')
differences=(fresh.Prediction != old.loc[fresh.index,'Prediction'].map(normalize))
print('Fresh greedy versus saved greedy: changed predictions =',int(differences.sum()))
print('Fresh minus saved greedy proxy =',summary.loc[1,'zindi_proxy']-selected['metrics']['zindi_score_proxy'])
print('A nonzero difference requires checking runtime/decoding reproducibility before interpreting small gains.')

## Paired wins/losses, hard groups and uncertainty

Differences are paired on identical images. Positive proxy delta means beam-four improves the local score. The 95% percentile bootstrap resamples validation images, recomputing both scores with the same sampled indices. This is exploratory: the development set already selected the checkpoint, and writer/page dependence is not accounted for.

In [ ]:
greedy=results[1].set_index('ID').loc[validation.ID].reset_index()
beam=results[4].set_index('ID').loc[validation.ID].reset_index()
assert greedy.Target.tolist()==beam.Target.tolist()
paired=greedy[['ID','Target','Prediction','word_edits','char_edits']].rename(columns={
    'Prediction':'greedy_prediction','word_edits':'greedy_word_edits','char_edits':'greedy_char_edits'})
paired['beam4_prediction']=beam.Prediction
paired['beam4_word_edits']=beam.word_edits;paired['beam4_char_edits']=beam.char_edits
ww=np.sqrt(greedy.nwords);cw=np.sqrt(greedy.nchars)
paired['beam4_proxy_gain']=.5*((greedy.word_edits-beam.word_edits)*ww/ww.sum()/WORD_NORMALIZER+
    (greedy.char_edits-beam.char_edits)*cw/cw.sum()/CHARACTER_NORMALIZER)
delta=summary.loc[4,'zindi_proxy']-summary.loc[1,'zindi_proxy']
assert np.isclose(paired.beam4_proxy_gain.sum(),delta)
print('Beam4: fewer/equal/more character errors:',int((beam.char_edits<greedy.char_edits).sum()),
    int((beam.char_edits==greedy.char_edits).sum()),int((beam.char_edits>greedy.char_edits).sum()))
display(paired.nlargest(10,'beam4_proxy_gain'));display(paired.nsmallest(10,'beam4_proxy_gain'))
paired.to_csv(OUT/'paired_changes.csv',index=False)

vocab=Counter(w.casefold().strip('.,;:!?"()') for text in audit.loc[audit.split.eq('train'),'Target'] for w in text.split())
slices=[]
for b,frame in results.items():
    frame=frame.copy()
    frame['has_caret']=frame.Target.str.contains('^',regex=False)
    frame['unseen_word']=frame.Target.map(lambda t:any(vocab[w.casefold().strip('.,;:!?"()')]==0 for w in t.split()))
    for column in ['label_group','height_band','has_caret','unseen_word']:
        for value,group in frame.groupby(column):
            slices.append({'num_beams':b,'grouping':column,'group':str(value),**summarize(group)})
slices=pd.DataFrame(slices)
display(slices[['num_beams','grouping','group','images','zindi_proxy','CER_percent','new_repetition_flags']])
slices.to_csv(OUT/'group_comparison.csv',index=False)

rng=np.random.default_rng(42)
indices=rng.integers(0,len(greedy),size=(BOOTSTRAP_REPLICATES,len(greedy)))
sample_ww=ww.to_numpy()[indices];sample_cw=cw.to_numpy()[indices]
word_difference=(greedy.word_edits-beam.word_edits).to_numpy()[indices]
char_difference=(greedy.char_edits-beam.char_edits).to_numpy()[indices]
bootstrap_delta=.5*((word_difference*sample_ww).sum(axis=1)/sample_ww.sum(axis=1)/WORD_NORMALIZER+
    (char_difference*sample_cw).sum(axis=1)/sample_cw.sum(axis=1)/CHARACTER_NORMALIZER)
low,high=np.quantile(bootstrap_delta,[.025,.975])
print(f'Beam4 proxy gain: {delta:+.6f}; exploratory paired 95% interval [{low:+.6f}, {high:+.6f}]')
print(f'Beam4 measured generation-time multiplier: {summary.loc[4,"relative_generation_time"]:.2f}x')
winner=4 if delta>0 else 1
decision={'best_checkpoint':SPEC,'highest_proxy_num_beams':winner,'beam4_proxy_gain':float(delta),
    'paired_bootstrap_95pct':[float(low),float(high)],'replicates':BOOTSTRAP_REPLICATES,
    'beam4_generation_time_multiplier':float(summary.loc[4,'relative_generation_time']),
    'experiment_key':EXPERIMENT_KEY,'greedy_saved_prediction_disagreements':int(differences.sum()),
    'note':'Highest proxy is not an automatic deployment decision; review uncertainty, failures and time.'}
write_json(OUT/'comparison_result.json',decision)
print('Highest measured proxy uses num_beams =',winner)
print('No training-run selection or submission has been changed. Reports:',OUT)

## How to use the result

Consider beam-four only if its measured gain is worthwhile relative to its latency and failure counts. If the paired interval includes zero, the holdout provides weak evidence for a reliable gain. Inspect regressions on names and superscripts before deciding. Repeated tuning on this same holdout is exploratory and can overfit the selection process.

The comparison result is kept separate from the training run's `final_selection.json`. Applying a chosen decoder to test/submission generation is a separate step.